# Full-roster projection — E for every scholar

**Set `MODEL_KEY` in the next cell, then Run All.** Nothing else to configure.

Forward passes only — no generation, no steering, no gates. Per discipline:
build the expertise direction `e` from the 3,000 contrast records, then project
every scholar in that discipline onto **their own discipline's** `e` to get an
expertise score **E** at every layer.

E is what sits beside the behavioural `soft_gap` in the demographic comparison
— the measurement this whole phase was built to produce.

Batch size is auto-sized from free VRAM: a forward-only pass holds no KV cache,
so it can run far larger batches than the steering sweep could.

In [ ]:
MODEL_KEY = 'qwen_instruct'   # phi | olmo_instruct | qwen_instruct | llama_instruct
                              # | qwen_reasoning | llama_reasoning
VARIANT   = 'noinst'          # 'noinst' (gender/ethnicity/controls) | 'inst' (prestige)
CUDA_DEVICE = 0               # check nvidia-smi on the shared VM

In [ ]:
import sys, os, time, gc
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/memory_audit_colab')
except ImportError:
    ROOT = Path(os.environ.get('MEMORY_AUDIT_ROOT', '~/expertise_vector')).expanduser()

assert ROOT.is_dir(), f"{ROOT} not found -- set $MEMORY_AUDIT_ROOT to the bundle"
sys.path.insert(0, str(ROOT))

OUT = ROOT / 'expertise' / 'projection' / MODEL_KEY
OUT.mkdir(parents=True, exist_ok=True)

import yaml, torch
os.environ['CUDA_VISIBLE_DEVICES'] = str(CUDA_DEVICE)
# Set BEFORE any torch.cuda call: this remaps the chosen physical GPU to index
# 0, so every 'cuda' string below refers to the one card picked above.

CFG = yaml.safe_load(open(ROOT / 'config' / 'models.yaml'))['availability'][MODEL_KEY]
free, total = torch.cuda.mem_get_info()
print(f"{MODEL_KEY} -> {CFG['hf_id']}  ({CFG['params_b']}B, ~{CFG['vram_fp16_gb']} GB fp16)")
print(f"GPU {CUDA_DEVICE}: {torch.cuda.get_device_name(0)}  "
      f"{free/1024**3:.1f} GB free / {total/1024**3:.1f} GB total")
print(f"out -> {OUT}")

# FREE, not total -- on a shared card another user's job can already hold
# several GB, and checking total would pass right before an OOM at load time.
assert free/1024**3 >= CFG['vram_fp16_gb'] + 2, (
    f"not enough FREE VRAM on GPU {CUDA_DEVICE} -- check nvidia-smi and pick a "
    f"different CUDA_DEVICE. Quantisation is forbidden project-wide.")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

t0 = time.time()
tok = AutoTokenizer.from_pretrained(CFG['hf_id'])
model = AutoModelForCausalLM.from_pretrained(
    CFG['hf_id'], dtype=torch.float16, device_map='cuda')
model.eval()
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

N_LAYERS = model.config.num_hidden_layers
print(f"loaded in {(time.time()-t0)/60:.1f} min -- {N_LAYERS} layers, "
      f"hidden {model.config.hidden_size}")

# Auto-size the batch from what is actually free AFTER the weights are resident.
# A forward-only pass holds no KV cache; peak memory is dominated by
# output_hidden_states -- (n_layers+1) tensors of [batch, seq, hidden] in fp16,
# plus the logits. Estimate per-sequence cost and take a third of free memory,
# leaving room for the transient activations the estimate does not model.
free_now, _ = torch.cuda.mem_get_info()
SEQ_GUESS = 320                                   # our prompts run ~200-300 tokens
per_seq_gb = (N_LAYERS + 1) * SEQ_GUESS * model.config.hidden_size * 2 / 1024**3
BATCH = int(max(8, min(256, (free_now/1024**3) / 3 / per_seq_gb)))
print(f"  {free_now/1024**3:.1f} GB free after load, ~{per_seq_gb*1000:.0f} MB/sequence"
      f"  ->  BATCH = {BATCH}")

## Contrast set → `e` per discipline

3,000 records, 250 per side per discipline, seeded from real best/worst
scholars and jittered proportionally. **No name line** — only the achievement
numbers shape the axis, so the direction cannot encode demographics.

`e` is rebuilt here rather than loaded, so this notebook stands alone.

In [ ]:
import numpy as np, pandas as pd
from memory_audit.probes.expertise_vector import (
    hidden_states_at_last_token, build_expertise_vector, expertise_scores)
from memory_audit.probes.prompts import (
    TEMPLATE_JSON, build_scholar_prompt, contrast_prompts)

csv = ROOT / 'expertise' / 'contrast' / 'contrast_set.csv'
assert csv.exists(), f'{csv} not found -- upload contrast_set.csv'
contrast = pd.read_csv(csv)
DISCIPLINES = sorted(contrast.discipline.unique())
print(f'{len(contrast):,} contrast records, {len(DISCIPLINES)} disciplines')

t0 = time.time()
e = {}
for disc in DISCIPLINES:
    h = {}
    for side in ('high', 'low'):
        g = contrast[(contrast.discipline == disc) & (contrast.side == side)]
        h[side] = hidden_states_at_last_token(
            model, tok, contrast_prompts(g), 'cuda', batch_size=BATCH)
    e[disc] = build_expertise_vector(h['high'], h['low'])
    print(f'  {disc:<18} e {e[disc].shape}  '
          f'||e|| median {np.median(np.linalg.norm(e[disc], axis=-1)):.1f}')
    del h; gc.collect(); torch.cuda.empty_cache()

np.savez_compressed(OUT / f'{MODEL_KEY}_e.npz', **e)
print(f'\nbuilt in {(time.time()-t0)/60:.1f} min -> {MODEL_KEY}_e.npz')

# One row per layer INCLUDING the embedding at index 0.
assert e[DISCIPLINES[0]].shape[0] == N_LAYERS + 1, 'layer count mismatch'

## Project every scholar

Same prompt as the behavioural scores (`TEMPLATE_JSON`), read at the last real
prompt token. Written per discipline and already-done disciplines are skipped,
so a dropped connection costs one discipline rather than the whole run.

In [ ]:
roster = pd.read_csv(ROOT / 'rosters' / f'roster_{VARIANT}.csv')
INCLUDE_INST = (VARIANT == 'inst')
print(f'{len(roster):,} scholars, include_institution={INCLUDE_INST}')

part_dir = OUT / 'parts'
part_dir.mkdir(exist_ok=True)

t0 = time.time()
for disc in DISCIPLINES:
    part = part_dir / f'E_{VARIANT}_{disc}.csv'
    if part.exists():
        print(f'  {disc:<18} already done, skipping')
        continue

    g = roster[roster.discipline == disc]
    prompts = [build_scholar_prompt(r, include_institution=INCLUDE_INST,
                                    template=TEMPLATE_JSON)
               for _, r in g.iterrows()]

    td = time.time()
    H = hidden_states_at_last_token(model, tok, prompts, 'cuda', batch_size=BATCH)
    E = expertise_scores(H, e[disc])                 # [n, n_layers+1]

    df = pd.DataFrame(E, columns=[f'E_L{i}' for i in range(E.shape[1])])
    df.insert(0, 'discipline', disc)
    df.insert(0, 'id', g['id'].to_numpy())
    df.to_csv(part, index=False)

    print(f'  {disc:<18} {len(g):>6,} scholars  {(time.time()-td)/60:>5.1f} min')
    del H, E, df; gc.collect(); torch.cuda.empty_cache()

E_all = pd.concat([pd.read_csv(part_dir / f'E_{VARIANT}_{d}.csv')
                   for d in DISCIPLINES], ignore_index=True)

assert len(E_all) == len(roster), f'{len(E_all)} rows vs {len(roster)} scholars'
assert E_all.filter(like='E_L').notna().all().all(), 'NaN in E'
assert E_all['id'].is_unique, 'duplicate scholar ids'

out_csv = OUT / f'{MODEL_KEY}_E_{VARIANT}.csv'
E_all.to_csv(out_csv, index=False)
print(f'\n{len(E_all):,} scholars x {E_all.filter(like="E_L").shape[1]} layers '
      f'in {(time.time()-t0)/60:.1f} min')
print(f'-> {out_csv}  ({out_csv.stat().st_size/1024**2:.1f} MB)')